# First Pass Yield
This notebook creates a first pass yield report for test results. It ties into the **Test Monitor Service** for retrieving filtered test results, the **Notebook Execution Service** for running outside of Jupyterhub, and the **Test Monitor Reports page** at #testmonitor/reports for displaying results.

The parameters and output use a schema recognized by the Test Monitor Reports page, which can be implemented by various report types. The First Pass Yield notebook produces data that is best shown in a bar graph.

### Imports
Import Python modules for executing the notebook. Pandas is used for building and handling dataframes. Scrapbook is used for recording data for the Notebook Execution Service. The SystemLink Test Monitor Client provides access to test result data for processing.

In [ ]:
import copy
import datetime
import pandas as pd
import scrapbook as sb
from dateutil import tz

import systemlink.clients.nitestmonitor as testmon

### Parameters
- `results_filter`: Dynamic Linq query filter for test results from the Test Monitor Service
  Options: Any valid Test Monitor Results Dynamic Linq filter
  Default: `'startedWithin <= "30.0:0:0"'`
- `group_by`: The dimension along which to reduce; what each bar in the output graph represents  
  Options: Day, System, Test Program, Operator, Part Number  
  Default: Day

Parameters are also listed in the metadata for the parameters cell, along with their default values. The Notebook Execution services uses that metadata to pass parameters from the Test Monitor Reports page to this notebook. Available `group_by` options are listed in the metadata as well; the Test Monitor Reports page uses these to validate inputs sent to the notebook.

To see the metadata, select the code cell and click the wrench icon in the far left panel.

In [ ]:
results_filter = 'startedWithin <= "28.0:0:0"'
products_filter = ''
group_by = 'Test Program'

# External Filters
workspace_filter = ''
product_filter = ''
testProgram_filter = ''
iteration_filter = ""
chart_type = ''
orientation = ''
isnum = True
iteration = ''

In [ ]:
if iteration_filter.isdigit():
    iteration = int(iteration_filter)
    isnum = True
else:
    isnum = False

In [ ]:
print(iteration)
print(isnum)


False


### Mapping from grouping options to Test Monitor terminology
Translate the grouping options shown in the Test Monitor Reports page to keywords recognized by the Test Monitor API.

In [ ]:
groups_map = {
    'Day': 'started_at',
    'System': 'host_name',
    'Test Program': 'program_name',
    'Operator': 'operator',
    'Part Number': 'part_number',
    'Workspace': 'workspace'
}
grouping = groups_map[group_by]

### Create Test Monitor client
Establish a connection to SystemLink over HTTP.

In [ ]:
results_api = testmon.ResultsApi()

In [ ]:
if workspace_filter:
    results_filter += f' && workspace = \"{workspace_filter}\"'
if product_filter:
    results_filter += f' && partNumber = \"{product_filter}\"'
if testProgram_filter:
    results_filter += f' && programName = \"{testProgram_filter}\"'
if isnum:
    results_filter += f' && properties[\"Test Iteration\"] = "{iteration}"'
print(results_filter)

startedWithin <= "28.0:0:0" && workspace = "a67109d9-0ba8-4c47-9612-a7863bf08f07" && partNumber = "TCU2" && programName = "TCU2-Main.seq" 


### Query for results
Query the Test Monitor Service for results matching the `results_filter` parameter.

In [ ]:
results_query = testmon.ResultsAdvancedQuery(
    results_filter, product_filter=products_filter, order_by=testmon.ResultField.STARTED_AT)


results = []
response = await results_api.query_results_v2(post_body=results_query)
results.extend(response.results)

while response.continuation_token:
    results_query.continuation_token = response.continuation_token
    response = await results_api.query_results_v2(post_body=results_query)
    results.extend(response.results)

results_list = [result.to_dict() for result in results]
#results_list

### Get group names
Collect the group name for each result based on the `group_by` parameter.

In [ ]:
group_names = []
for result in results_list:
    group_names.append(result.get(grouping, ''))

### Create pandas dataframe
Put the data into a dataframe whose columns are serial number, status, start time, and group name. Sort and group the dataframe to get the first test run for each unique serial number.

In [ ]:
formatted_results = {
    'id': [result['id'] for result in results_list],
    'serial_number': [result['serial_number'] for result in results_list],
    'status': [result['status']['status_type'] for result in results_list],
    'started_at': [result['started_at'] for result in results_list],
    'updated_at': [result['updated_at'] for result in results_list],
    grouping: group_names
}

df_results = pd.DataFrame.from_dict(formatted_results)

if grouping == 'started_at':
    sorting_list = ['serial_number', 'started_at']
    grouping_list = ['serial_number']

else:
    sorting_list = [grouping, 'serial_number', 'started_at']
    grouping_list = [grouping, 'serial_number']

df_results = df_results.sort_values(by=sorting_list)
#df_results_unique = df_results.groupby(grouping_list).last().reset_index()
df_results_unique = df_results
#df_results_unique

### Handle grouping by day
If the grouping is by day, the group name is the date and time when the test started in UTC. To group all test results from a single day together, convert to server time and remove time information from the group name.

In [ ]:
df_results_copy = copy.copy(df_results_unique)
df_results_copy.fillna(value='', inplace=True)

if grouping == 'started_at':
    truncated_times = []
    for val in df_results_copy[grouping]:
        local_time = val.astimezone(tz.tzlocal())
        truncated_times.append(str(datetime.date(local_time.year, local_time.month, local_time.day)))
    df_results_copy[grouping] = truncated_times
#print(df_results_copy.to_string())

### Aggregate results into groups
Aggregate the data for each unique group and status.

*See documentation for [size](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.size.html) and [unstack](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.unstack.html) here.*

In [ ]:
df_grouped = df_results_copy.groupby([grouping, 'status']).size().unstack(fill_value=0)
if 'PASSED' not in df_grouped:
    df_grouped['PASSED'] = 0
if 'FAILED' not in df_grouped:
    df_grouped['FAILED'] = 0
if 'ERRORED' not in df_grouped:
    df_grouped['ERRORED'] = 0
if 'TERMINATED' not in df_grouped:
    df_grouped['TERMINATED'] = 0
if 'TIMED_OUT' not in df_grouped:
    df_grouped['TIMED_OUT'] = 0
if 'RUNNING' not in df_grouped:
    df_grouped['RUNNING'] = 0
df_grouped

#We can check here also for a 0 dataframe


status,FAILED,PASSED,TERMINATED,ERRORED,TIMED_OUT,RUNNING
program_name,,,,,,
TCU2-Main.seq,151,303,7,0,0,0


### First Pass Yield calculation
Divide the number of passed tests by the total number of tests.

In [ ]:
df_first_pass_yield = []
print(df_first_pass_yield)
df_first_pass_yield = pd.DataFrame(100 * df_grouped['PASSED'] / (df_grouped['FAILED'] + df_grouped['ERRORED'] + df_grouped['PASSED'] + df_grouped['TERMINATED']+ df_grouped['TIMED_OUT']))
df_first_pass_yield['1'] = pd.DataFrame((df_grouped['FAILED'] + df_grouped['ERRORED'] + df_grouped['PASSED'] + df_grouped['TERMINATED']+ df_grouped['TIMED_OUT']))
df_first_pass_yield['2'] = pd.DataFrame(df_grouped['PASSED'])
df_first_pass_yield['3'] = pd.DataFrame(df_grouped['FAILED'])
df_first_pass_yield['4'] = pd.DataFrame(df_grouped['RUNNING'])
df_first_pass_yield['5'] = pd.DataFrame(df_grouped['ERRORED'] + df_grouped['TERMINATED'] + df_grouped['TIMED_OUT'])
df_first_pass_yield = df_first_pass_yield.reset_index().set_axis([grouping, 'Yield', 'Total', 'Passed', 'Failed', 'Running', 'Other'], axis=1)

[]


,program_name,yield,Total,Passed,Failed,Running,Other
0,TCU2-Main.seq,65.726681,461,303,151,0,7


### Convert the dataframe to the SystemLink reports output format
The result format for a SystemLink report consists of a list of output objects as defined below:
- `type`: The type of the output. Accepted values are 'data_frame' and 'scalar'.
- `id`: Corresponds to the id specified in the 'output' metadata. Used for returning multiple outputs with the 'V2' report format.
- `data`: A dict representing the 'data_frame' type output data.
    - `columns`: A list of dicts containing the names and data type for each column in the dataframe.
    - `values`: A list of lists containing the dataframe values. The sublists are ordered according to the 'columns' configuration.
- `value`: The value returned for the 'scalar' output type.
- `config`: The configurations for the given output.
    - `title`: The output title.
    - `graph`: The graph configurations.
        - `axis_labels`: The x-axis label and y-axis label.
        - `plots`: A list of plots to display mapped from the dataframe's columns, along with configuration options.
            - `x`: The dataframe column corresponding to the x-axis values.
            - `y`: The dataframe column corresponding to the y-axis values.
            - `style`: The plot's style. Accepted values are ['LINE', 'BAR', 'SCATTER'].
            - `color`: The plot's color. Accepted formats are ['blue', '#0000ff', 'rbg(0,0,255)'].
            - `label`: The plot's name, to be shown in a plot legend. 
            - `secondary_y`: Whether or not to display this plot on a second y-axis.
            - `group_by`: A list of columns in the dataframe on which to group data, e.g. to color individual points.
        - `orientation`: 'HORIZONTAL' or 'VERTICAL'.
        - `stacked`: Whether or not to display the plots stacked on top of each other.

Here is an example of a notebook result with two outputs, one of which is a dataframe with two columns, and the other is a scalar value:
```
[{
    'type': 'data_frame',
    'id': 'output_id_1',
    'data': {
        'columns': [
            {'name': 'time', 'type': 'datetime'},
            {'name': 'value', 'type': 'number'}
         ],
        'values': [
            ['2020-09-29T00:00:00.000Z', 46.1538461538],
            ['2020-09-30T00:00:00.000Z', 63.1578947368],
            ...
         ]
    },
    'config': {
        'title': 'My Title',
        'graph': {
            'axis_labels': ['X Axis', 'Y Axis'],
            'orientation': 'VERTICAL',
            'plots': [
                {'x': 'time', 'y': 'value', 'style': 'BAR', 'color': '#0000ff', 'label': 'Plot 1'}
            ]
        }
    }
}, {
    'type': 'scalar',
    'id': 'output_id_2',
    'config': {
        'title': 'My Title'
    },
    'value': 5
}]
```

For this report, there is one output, which is a dataframe with two columns. For a grouping of 'Day', the first column contains ISO-8601 date strings. For any other grouping option, the first column contains categorical string values. The second column contains numerical values representing the yield percentages.

| started_at                 | yield         |
|----------------------------|---------------|
| '2020-09-29'               | 46.1538461538 |
| '2020-09-30'               | 63.1578947368 |
| '2020-10-01'               | 60.0          |

The graph configuration specifies a single plot, where the x-axis is the group values and the y-axis is the yield. We use Pandas to convert the dataframe built in the previous cells into a tabular format and then return that with the result object.

In [ ]:
df_dict = {
    'columns': pd.io.json.build_table_schema(df_first_pass_yield, index=False)['fields'],
    'values': df_first_pass_yield.values.tolist(),
}

totals_table = {
    'type': 'data_frame',
    'id': 'df_first_pass_yield',
    'data': df_dict
}

result = [totals_table]

### Record results with Scrapbook

In [ ]:
sb.glue('result', result)
result

[{'type': 'data_frame',
  'id': 'df_first_pass_yield',
  'data': {'columns': [{'name': 'program_name', 'type': 'string'},
    {'name': 'yield', 'type': 'number'},
    {'name': 'Total', 'type': 'integer'},
    {'name': 'Passed', 'type': 'integer'},
    {'name': 'Failed', 'type': 'integer'},
    {'name': 'Running', 'type': 'integer'},
    {'name': 'Other', 'type': 'integer'}],
   'values': [['TCU2-Main.seq', 65.72668112798264, 461, 303, 151, 0, 7]]}}]